# Step 6 — Allen atlas transformation

**Purpose.** Estimate one atlas-to-image transform per `group_id` + `ref_stem`, project
Allen dorsal-surface regions to the Step3 anatomical reference, and optionally
produce processed-space atlas labels and regional coverage for every selected stem.

**Inputs.** Step1 manifest, Step2 registration/masks, Step3 reference annotations
and optional manual registration overrides, Step5 response ROI masks, an Allen
BrainGlobe atlas, and an atlas-landmarks CSV (created interactively when absent).
A Step4 processed Zarr is used when available to determine each stem's spatial
shape; the configured fallbacks are documented below.

**Outputs.** A Step6 index, reference-space response masks, optional population
response templates, fitted transform JSON files, reference and stem label maps,
region coverage CSVs, atlas overlays and processing logs. Output folder names
are set under `step6.output` in the project YAML.

Use the working YAML through `CONFIG_PATH`. Keeping a copy of the final YAML with
the analysis is recommended for reproducibility, but the code does not require
a particular config directory.


**Cohort and registration policy.** Step6 is an anatomical projection stage,
not the step that applies scientific cohort exclusions. By default it retains
anatomically usable Step2/Step3 entries irrespective of experimental group,
`UNASSIGNED`, REST and `analysis_exclusions`; scientific exclusions are applied
in Step6b. Explicit `step6.batch` filters, `step6.batch.include_bad`, and
`step6.reference.require_step3_annotation` can still restrict Step6 rows.

When a full 3×3 manual Step3 registration transform is available, it takes
precedence over automatic Step2 registration. A real Step5 response ROI is
transferred to Step3 reference coordinates before determining its hemisphere.
Without an independently specified side, a missing response anchor is **not**
replaced with a population template. The fit can then rely on anatomical
landmarks, subject to `step6.fit.allow_fit_without_response_mask`.

The notebook's single-group diagnostic and full-atlas overlay cells are
preserved. Start with a fresh Jupyter kernel and run the required cells in order;
see the explicit exception for saved atlas landmarks below.


## Response-anchor selection

The selection priority follows `step6.response_anchor.policy` and its associated
flags:

1. `ref_stem_response_roi`: a Step5 mask on the reference recording.
2. `same_group_registered_response_roi`: a mask from another stem in the same
   `group_id`, registered into reference space (if permitted).
3. `population_ref_stem_response_template`: a fallback only when the side is
   explicitly verified by `step6.missing_response_masks.side_policy: fixed`
   and a valid `fixed_side` (`left` or `right`) is configured.
4. No response anchor: anatomical-only fitting when permitted.

For real ROIs, the detected side is determined geometrically from the annotated
AP axis and settings under `step6.side_detection`; it is not inferred from the
recording protocol. Template construction and use are distinct: creating donor
templates does not authorize using them for a group with an unknown side.


In [ ]:
from pathlib import Path
import sys
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Expected layout: project/notebook/Step6_AtlasTransformation.ipynb
#                  project/src/common.py and step6_atlas_transformation.py
# Run this notebook with project/notebook/ as the Jupyter working directory.
NOTEBOOK_DIR = Path.cwd().resolve()
SRC_DIR = (NOTEBOOK_DIR.parent / "src").resolve()

required_files = ("common.py", "step6_atlas_transformation.py")
missing_files = [name for name in required_files if not (SRC_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Required source files were not found: {', '.join(missing_files)}. "
        f"Expected src/ location: {SRC_DIR}. "
        "Run this notebook from the project's notebook/ directory."
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from step6_atlas_transformation import *
from step6_atlas_transformation import _label_boundaries, _reference_row_for_group
print('Source folder:', SRC_DIR)

In [ ]:
# Example project-specific path; replace CONFIG_PATH with your working YAML.
# Keeping a copy of that YAML with the analysis is recommended.
# Atlas name and atlas landmark filename are read from:
#   step6.atlas.name
#   step6.atlas.landmarks_csv_name
CONFIG_PATH = r"config.yaml"

config, paths = standard_project_paths(CONFIG_PATH)
paths


## 1. Build the Step6 index

`build_step6_index()` combines `manifest_recordings.csv`,
`metadata_registration.csv`, Step3 `reference_annotations.csv`, optional Step3
manual registration overrides, Step5 response ROI availability and Step4
processed-Zarr paths. It writes the configured `step6.output.step6_index_name`.

The index is **stem-level**; the fitted atlas transform is identified by the
pair `group_id` + `ref_stem`. The default requirement for complete Step3
annotations is controlled by `step6.reference.require_step3_annotation`.


In [ ]:
step6_index, config, paths = build_step6_index(
    CONFIG_PATH,
    require_step3_annotation=True,
    require_response_roi=False,
)

cols = [c for c in [
    "stem", "group_id", "ref_stem", "exp_day", "state", "protocol",
    "step3_annotation_complete", "response_roi_exists", "ref_image_exists", "processed_zarr_exists"
] if c in step6_index.columns]
step6_index[cols]


## 2. Check response masks and hemisphere assignment

This diagnostic does not need the Allen atlas. It converts available Step5
`.npy` response masks from stem/processed resolution into the Step3 reference
frame and checks hemisphere classification relative to the reference AP axis.

Transforms use final manual Step3 registration when available; otherwise they
use Step2 registration. The geometry is in full-resolution reference pixels,
with physical ML/AP offsets derived from `step6.geometry.pixel_size_um`.


In [ ]:
# Prefer a group where response ROI exists on the ref_stem itself; if absent, same-group non-ref ROI can be registered to ref_stem.
ref_response_mask = (
    step6_index["response_roi_exists"].astype(bool)
    & step6_index["stem"].astype(str).eq(step6_index["ref_stem"].astype(str))
)
if ref_response_mask.any():
    row_test = step6_index[ref_response_mask].iloc[0]
elif step6_index["response_roi_exists"].any():
    # This lets us inspect a group where only non-reference response exists.
    # The workflow will transform that non-ref mask into the reference frame before using it.
    row_test = step6_index[step6_index["response_roi_exists"]].iloc[0]
else:
    row_test = step6_index.iloc[0]

row_test[[c for c in ["stem", "group_id", "ref_stem", "protocol", "response_roi_exists", "response_roi_path", "ref_image_path"] if c in row_test.index]]


In [ ]:
# Build side-specific donor templates; when side_policy='missing' they are diagnostic only, NOT automatically substituted.
response_templates = build_reference_response_population_templates(
    step6_index,
    config,
    paths,
    save=True,
)
print("Population templates:", {k: v.get("n_masks", 0) for k, v in response_templates.items()})

# Resolve the response anchor for the selected group/ref_stem.
group_key = (row_test["group_id"], row_test["ref_stem"])
group_df = step6_index[
    (step6_index["group_id"].astype(str) == str(group_key[0])) &
    (step6_index["ref_stem"].astype(str) == str(group_key[1]))
].reset_index(drop=True)

row0 = _reference_row_for_group(group_df)
ref_img = load_reference_image_for_row(row0)
ref_shape = ref_img.shape[:2]
basis = reference_basis_from_row(row0, config)
mask_ref, response_info, response_ml_ap_um = resolve_response_anchor_for_group(
    group_df,
    config,
    paths,
    row0,
    ref_shape,
    basis,
    response_templates=response_templates,
)

print(response_info)
if mask_ref is not None:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(normalize01(ref_img), cmap="gray" if ref_img.ndim == 2 else None)
    ax.contour(mask_ref.astype(float), levels=[0.5], colors="yellow", linewidths=1.2)
    origin = basis["origin_xy"]
    ax.plot([origin[0], origin[0] + 120*basis["ap_hat_xy"][0]], [origin[1], origin[1] + 120*basis["ap_hat_xy"][1]], "r-", label="AP+")
    ax.plot([origin[0], origin[0] + 120*basis["ml_hat_xy"][0]], [origin[1], origin[1] + 120*basis["ml_hat_xy"][1]], "b-", label="ML+")
    ax.scatter([basis["bregma_xy"][0]], [basis["bregma_xy"][1]], c="red", s=40, label="image bregma")
    ax.scatter([basis["rostral_xy"][0]], [basis["rostral_xy"][1]], c="lime", s=40, label="image rostral")
    ax.axis("off")
    ax.legend();
else:
    print("No response anchor for this reference group.")


In [ ]:
# Batch-convert all available response masks to reference space and save side-detection log.
side_df = process_response_masks_to_reference(step6_index, config, paths, save=True)
side_df.head()


## 3. Load the Allen atlas

This stage requires `brainglobe-atlasapi` and the atlas specified by
`step6.atlas.name`. If the package or configured atlas data are absent, install
or make them available in the active Python environment before running this cell.
For example, the package installation command is:

```shell
python -m pip install brainglobe-atlasapi
```

The atlas name and directory are configurable; a particular atlas name is **not**
a universal requirement of this workflow. The code loads the atlas with
`check_latest=False` and constructs its dorsal (AP, ML) projection.


In [ ]:
atlas_data = load_allen_atlas(config)
print("Atlas loaded")
print("flat label shape:", atlas_data.flat_label.shape)
print("resolution AP/DV/ML µm:", atlas_data.res_ap_um, atlas_data.res_dv_um, atlas_data.res_ml_um)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(atlas_data.flat_ref_disp, cmap="gray")
ax.contour(_label_boundaries(atlas_data.flat_label).astype(float), levels=[0.5], colors="white", linewidths=0.3)
ax.set_title("Allen dorsal flat projection: AP x ML")
ax.axis("off");


## 4. Annotate atlas landmarks once

Two atlas landmark layers are required in Napari: `bregma_atlas` and
`rostral_atlas`. Lambda is **not** part of the Step6 atlas fit. The landmark
filename is specified in `step6.atlas.landmarks_csv_name` and the atlas itself
by `step6.atlas.name` in the project YAML.

**First run / CSV absent:** run the next cell to open Napari, place exactly one
point per layer, close the viewer, **then run the following save cell**
(`save_atlas_landmarks_from_napari_session(atlas_session)`).

**Saved landmarks CSV exists:** the next cell loads it and does **not** create
`atlas_session`. **Skip the following save-only cell** and continue with the
single-group fit. Running that save cell with a clean kernel otherwise raises
`NameError`; with a reused kernel it may reference an outdated session.

The later optional reopen cell is for intentionally revising atlas landmarks.
After closing its viewer, run the save cell only when you intend to overwrite
previously saved landmarks. Existing landmarks are not automatically rewritten.


In [ ]:
atlas_landmarks_csv = Path(paths["atlas_landmarks_csv"])
if atlas_landmarks_csv.exists():
    print("Using saved atlas landmarks:", atlas_landmarks_csv)
    atlas_lm_df = pd.read_csv(atlas_landmarks_csv)
    # A saved CSV does not create atlas_session; skip the next save-only cell.
else:
    atlas_session = open_atlas_landmark_napari_viewer(atlas_data, paths, config)
    # After the viewer opens, place bregma_atlas and rostral_atlas, then close napari.
    napari.run()

In [ ]:
atlas_lm_df = save_atlas_landmarks_from_napari_session(atlas_session)

atlas_lm_df

In [ ]:
# Re-run this cell only if you intentionally want to replace saved atlas landmarks.
atlas_session = open_atlas_landmark_napari_viewer(atlas_data, paths, config)
napari.run()
# atlas_lm_df = save_atlas_landmarks_from_napari_session(atlas_session)
# atlas_lm_df


## 5. Test one `group_id` / `ref_stem` atlas fit

This is a **real write operation**, not a read-only preview. It saves the
per-reference transform JSON, dorsal atlas label map and structure table,
optional overlay, and per-stem processed-resolution labels with coverage CSVs.

The fit follows `step6.fit`: stage-1 bounded pitch optimization and optional
bounded pitch/scale refinement. It matches atlas bregma/rostral landmarks
and, only when eligible, a response anchor. The reference AP/ML basis uses
image pixels and the configured µm/px scale; atlas coordinates use µm.


In [ ]:
group_key = (row_test["group_id"], row_test["ref_stem"])
group_df = step6_index[
    (step6_index["group_id"].astype(str) == str(group_key[0])) &
    (step6_index["ref_stem"].astype(str) == str(group_key[1]))
].reset_index(drop=True)

group_df[[c for c in ["stem", "group_id", "ref_stem", "protocol", "response_roi_exists", "processed_zarr_exists"] if c in group_df.columns]]


In [ ]:
summary = process_step6_reference(
    group_df,
    config,
    paths,
    atlas_data=atlas_data,
    save_stem_labelmaps=True,
    response_templates=response_templates,
    save_full_atlas_overlay=False,
)
summary


In [ ]:
# Show saved overlay from the test group.
from IPython.display import Image, display
if summary.get("overlay_path"):
    display(Image(filename=summary["overlay_path"]))


## 5.1 Inspect the full atlas extent

This optional QC visualization projects the configured atlas contours beyond
the boundaries of the reference image. Configurable regions include cortex,
olfactory bulbs, cerebellum and an optional response-target region; overlays can
also include the response mask and atlas/image landmarks.

Rendering options are in `step6.full_atlas_overlay`. The preview writes its
figure but does not modify the fitted transform or atlas label maps. The
subsequent code-cell `setdefault()` calls change missing options in the
**in-memory** configuration for this session; they do not update the YAML.


In [ ]:
# Full-size atlas extent overlay settings can be changed in config under:
# step6.full_atlas_overlay
# For this interactive check we call the renderer explicitly for the selected group.
config["step6"].setdefault("full_atlas_overlay", {})
config["step6"]["full_atlas_overlay"].setdefault("target_side_mode", "detected")
config["step6"]["full_atlas_overlay"].setdefault("show_projected_atlas_landmarks", True)
config["step6"]["full_atlas_overlay"].setdefault("show_image_landmarks", True)
config["step6"]["full_atlas_overlay"]


In [ ]:
# Render full-size atlas extent overlay for the test group.
# Requirements: summary, group_df, atlas_data, config, paths from the cells above.

row0 = _reference_row_for_group(group_df)
ref_img = load_reference_image_for_row(row0)
ref_shape = ref_img.shape[:2]
with open(summary["transform_path"], "r", encoding="utf-8") as f:
    transform = json.load(f)

basis = reference_basis_from_row(row0, config)
response_mask_ref, response_info_full, _ = resolve_response_anchor_for_group(
    group_df,
    config,
    paths,
    row0,
    ref_shape,
    basis,
    response_templates=response_templates,
)

fig, full_overlay_path, full_overlay_info = render_full_atlas_extent_overlay(
    ref_img,
    atlas_data,
    transform,
    row0,
    group_df,
    config,
    paths,
    response_mask_ref=response_mask_ref,
    detected_side=response_info_full.get("detected_side", "missing"),
)

print(full_overlay_path)
full_overlay_info


In [ ]:

from IPython.display import Image, display

display(Image(filename=str(full_overlay_path)))


## 6. Inspect regional coverage

For each projected atlas region, the coverage fraction is the number of pixels
retained by the usable `brain_signal` mask divided by the total number of
projected region pixels. The output also includes area counts and region status.

The usable mask follows the code's exact expression
`brain & ~bad & ~bregma`. Thresholds and the minimum region area are set in
`step6.coverage`. Coverage describes the usable portion of the **projected**
region, not an independent anatomical validation of the atlas fit.


In [ ]:
test_stem = str(group_df.iloc[0]["stem"])
coverage_path = paths["tables_root"] / f"{test_stem}_atlas_region_coverage.csv"
coverage = pd.read_csv(coverage_path)
coverage.sort_values("coverage_fraction", ascending=False).head(30)


## 7. Process all selected groups

Run the batch only after checking landmark placement and the trial pitch/scale
fit. The provided active cell invokes `run_step6_batch()` and **writes** atlas
products for the selected groups; it enables full-extent overlays explicitly.
To run only the alternative ordinary-overlay variant, see the commented
example in the next code cell. Selection and error handling are configured in
`step6.batch` and `step6.strict`.


In [ ]:
# Uncomment for full batch with ordinary overlay only.
# batch_summary = run_step6_batch(CONFIG_PATH, load_atlas=True, save_full_atlas_overlay=False)
# display(batch_summary)

# Active batch command: save full atlas extent overlays for each reference group.
batch_summary = run_step6_batch(CONFIG_PATH, load_atlas=True, save_full_atlas_overlay=True)
display(batch_summary)

# Inspect response-anchor source balance after batch.
# display(batch_summary[["group_id", "ref_stem", "response_anchor_source", "response_anchor_stem", "detected_side", "response_weight_multiplier", "full_atlas_overlay_path"]])


## CLI equivalents

Run these from the project's `src/` directory, replacing `CONFIG_PATH` with the
actual full YAML path. The paths shown below are examples, **not** required
locations:

```bat
python step6_atlas_transformation.py --config C:\example\analysis\config.yaml --list
python step6_atlas_transformation.py --config C:\example\analysis\config.yaml --prepare-response-refspace
python step6_atlas_transformation.py --config C:\example\analysis\config.yaml --batch
python step6_atlas_transformation.py --config C:\example\analysis\config.yaml --batch --full-atlas-overlay
```

Atlas landmark annotation is performed interactively in Napari; the Step6 CLI
does not provide an equivalent command for placing those two points. The batch
requires a valid saved atlas landmarks CSV. In the notebook, `CONFIG_PATH`
must likewise point to the user's working YAML.
